In [ ]:
!pip -q install datasets pandas huggingface_hub

In [ ]:
import json
import random
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login

In [ ]:
OUT_CSV = "prompts.csv"
N_PER_BENCH = 50
SEED = 1337

def sample_prompts(rows, prompt_key, n=50, seed=1337, meta_keys=None, id_key=None):
    meta_keys = meta_keys or []
    cleaned = []

    for i, r in enumerate(rows):
        p = r.get(prompt_key)
        if p is None:
            continue
        if not isinstance(p, str):
            p = str(p)
        p = p.strip()
        if not p:
            continue

        row_id = None
        if id_key and id_key in r:
            row_id = r[id_key]
        else:
            row_id = r.get("id", r.get("_id", i))

        cleaned.append({
            "row_id": row_id,
            "prompt": p,
            "meta": {k: r.get(k) for k in meta_keys if k in r},
        })

    # dedup by prompt
    dedup = {}
    for item in cleaned:
        dedup.setdefault(item["prompt"], item)
    unique = list(dedup.values())

    rng = random.Random(seed)
    rng.shuffle(unique)

    return unique[:n]

In [ ]:
def append_to_csv(records, out_csv=OUT_CSV):
    df = pd.DataFrame(records)
    # append mode with header only if file doesn't exist
    try:
        with open(out_csv, "r", encoding="utf-8"):
            exists = True
    except FileNotFoundError:
        exists = False

    df.to_csv(out_csv, mode="a", index=False, header=not exists, encoding="utf-8")
    print(f"Appended {len(df)} rows -> {out_csv}")

In [ ]:
def reset_csv(out_csv=OUT_CSV):
    import os
    if os.path.exists(out_csv):
        os.remove(out_csv)
        print(f"Removed existing {out_csv}")
    else:
        print(f"No existing {out_csv} to remove")

# reset_csv()

In [ ]:
login()  

In [ ]:
ds = load_dataset("JailbreakBench/JBB-Behaviors", 'judge_comparison')  
split = "test" if "test" in ds else ("train" if "train" in ds else list(ds.keys())[0])
rows = [dict(r) for r in ds[split]]

prompt_key = "prompt" if "prompt" in rows[0] else ("goal" if "goal" in rows[0] else None)
if prompt_key is None:
    raise ValueError(f"JBB: can't find prompt column. Columns: {list(rows[0].keys())}")

meta_keys = [k for k in ("goal", "human_majority", "category") if k in rows[0]]

sampled = sample_prompts(rows, prompt_key=prompt_key, n=N_PER_BENCH, seed=SEED, meta_keys=meta_keys)

records = [{
    "benchmark": "JailbreakBench (JBB)",
    "source": "hf:JailbreakBench/JBB-Behaviors",
    "split": split,
    "row_id": s["row_id"],
    "prompt": s["prompt"],
    "meta_json": json.dumps(s["meta"], ensure_ascii=False),
} for s in sampled]

append_to_csv(records)

In [ ]:
ds = load_dataset("walledai/StrongREJECT")
split = "test" if "test" in ds else ("train" if "train" in ds else list(ds.keys())[0])
rows = [dict(r) for r in ds[split]]

candidate_keys = [k for k in ("forbidden_prompt", "prompt", "query") if k in rows[0]]
if not candidate_keys:
    raise ValueError(f"StrongREJECT: can't find prompt column. Columns: {list(rows[0].keys())}")
prompt_key = candidate_keys[0]

meta_keys = [k for k in ("category", "domain", "answer") if k in rows[0]]

sampled = sample_prompts(rows, prompt_key=prompt_key, n=N_PER_BENCH, seed=SEED, meta_keys=meta_keys)

records = [{
    "benchmark": "StrongREJECT",
    "source": "hf:walledai/StrongREJECT",
    "split": split,
    "row_id": s["row_id"],
    "prompt": s["prompt"],
    "meta_json": json.dumps(s["meta"], ensure_ascii=False),
} for s in sampled]

append_to_csv(records)


In [ ]:
DATASET_ID = "walledai/AdvBench"  

ds = load_dataset(DATASET_ID)
split = "test" if "test" in ds else ("train" if "train" in ds else list(ds.keys())[0])
rows = [dict(r) for r in ds[split]]

candidate_keys = [k for k in ("prompt", "instruction", "goal", "query") if k in rows[0]]
if not candidate_keys:
    raise ValueError(f"AdvBench: can't find prompt column. Columns: {list(rows[0].keys())}")
prompt_key = candidate_keys[0]

meta_keys = [k for k in ("category", "source") if k in rows[0]]

sampled = sample_prompts(rows, prompt_key=prompt_key, n=N_PER_BENCH, seed=SEED, meta_keys=meta_keys)

records = [{
    "benchmark": "AdvBench",
    "source": f"hf:{DATASET_ID}",
    "split": split,
    "row_id": s["row_id"],
    "prompt": s["prompt"],
    "meta_json": json.dumps(s["meta"], ensure_ascii=False),
} for s in sampled]

append_to_csv(records)


In [ ]:
ds = load_dataset("allenai/wildjailbreak",  'eval')

split = "test" if "test" in ds else ("train" if "train" in ds else list(ds.keys())[0])
rows = [dict(r) for r in ds[split]]

# print("Split:", split)
# print("Columns:", list(rows[0].keys()))
# print("Example row:", {k: (str(rows[0][k])[:120] + "..." if isinstance(rows[0][k], str) and len(rows[0][k]) > 120 else rows[0][k]) for k in rows[0].keys()})

preferred = ["adversarial", "prompt", "instruction", "query", "text", "user_prompt"]
prompt_key = next((k for k in preferred if k in rows[0]), None)

def _lower(x):
    return str(x).lower()

if "data_type" in rows[0]:
    adv_rows = [r for r in rows if "adv" in _lower(r.get("data_type")) or "jail" in _lower(r.get("data_type"))]
    if len(adv_rows) >= N_PER_BENCH:
        rows = adv_rows
        print("Filtered by data_type ->", len(rows))

if "label" in rows[0]:
    harmful_rows = [r for r in rows if any(tok in _lower(r.get("label")) for tok in ["harm", "adv", "jail", "unsafe", "malicious"])]
    if len(harmful_rows) >= N_PER_BENCH:
        rows = harmful_rows
        print("Filtered by label ->", len(rows))

meta_keys = [k for k in ("label", "data_type") if k in rows[0]]

sampled = sample_prompts(rows, prompt_key=prompt_key, n=N_PER_BENCH, seed=SEED, meta_keys=meta_keys)

records = [{
    "benchmark": "WildJailbreak",
    "source": "hf:allenai/wildjailbreak",
    "split": split,
    "row_id": s["row_id"],
    "prompt": s["prompt"],
    "meta_json": json.dumps(s["meta"], ensure_ascii=False),
} for s in sampled]

append_to_csv(records)


In [ ]:
ds = load_dataset("walledai/HarmBench", "contextual")

split = "test" if "test" in ds else ("train" if "train" in ds else list(ds.keys())[0])
rows = [dict(r) for r in ds[split]]

# print("Split:", split)
# print("Columns:", list(rows[0].keys()))

if "prompt" not in rows[0]:
    raise ValueError(f"HarmBench contextual: missing 'prompt' column. Columns: {list(rows[0].keys())}")

combined_rows = []
for i, r in enumerate(rows):
    ctx = r.get("context", "")
    prm = r.get("prompt", "")
    if ctx is None: ctx = ""
    if prm is None: prm = ""
    ctx = str(ctx).strip()
    prm = str(prm).strip()
    if not ctx and not prm:
        continue

    combined_text = (ctx + "\n\n" + prm).strip() if ctx else prm

    combined_rows.append({
        **r,
        "_combined_prompt": combined_text,  
        "_row_id": r.get("id", r.get("_id", i)),
    })

meta_keys = [k for k in combined_rows[0].keys() if k not in ("_combined_prompt",)]

sampled = sample_prompts(
    combined_rows,
    prompt_key="_combined_prompt",
    n=N_PER_BENCH,
    seed=SEED,
    meta_keys=meta_keys,
    id_key="_row_id",
)

records = [{
    "benchmark": "HarmBench (contextual: context+prompt)",
    "source": "hf:walledai/HarmBench@contextual",
    "split": split,
    "row_id": s["row_id"],
    "prompt": s["prompt"],  # context + prompt
    "meta_json": json.dumps(s["meta"], ensure_ascii=False),
} for s in sampled]

append_to_csv(records)


In [ ]:
url = "https://raw.githubusercontent.com/paul-rottger/xstest/main/xstest_prompts.csv"
df = pd.read_csv(url)

if "prompt" not in df.columns:
    raise ValueError(f"XSTest: expected 'prompt' column. Columns: {list(df.columns)}")

rows = df.to_dict(orient="records")
meta_cols = [c for c in df.columns if c != "prompt"]

sampled = sample_prompts(rows, prompt_key="prompt", n=N_PER_BENCH, seed=SEED, meta_keys=meta_cols)

records = [{
    "benchmark": "XSTest",
    "source": "github:paul-rottger/xstest/xstest_prompts.csv",
    "split": "n/a",
    "row_id": s["row_id"],
    "prompt": s["prompt"],
    "meta_json": json.dumps(s["meta"], ensure_ascii=False),
} for s in sampled]

append_to_csv(records)


In [ ]:
url = "https://raw.githubusercontent.com/mlcommons/ailuminate/main/airr_official_1.0_demo_en_us_prompt_set_release.csv"
df = pd.read_csv(url)

text_cols = [c for c in df.columns if c.lower() in ("prompt", "prompt_text", "text", "user_prompt")]
if text_cols:
    prompt_col = text_cols[0]
else:
    obj_cols = [c for c in df.columns if df[c].dtype == "object"]
    if not obj_cols:
        raise ValueError("AILuminate: cannot find prompt-like column.")
    prompt_col = obj_cols[0]

rows = df.to_dict(orient="records")
meta_cols = [c for c in df.columns if c != prompt_col]

sampled = sample_prompts(rows, prompt_key=prompt_col, n=N_PER_BENCH, seed=SEED, meta_keys=meta_cols)

records = [{
    "benchmark": "MLCommons AILuminate (DEMO)",
    "source": "github:mlcommons/ailuminate/airr_official_1.0_demo_en_us_prompt_set_release.csv",
    "split": "n/a",
    "row_id": s["row_id"],
    "prompt": s["prompt"],
    "meta_json": json.dumps(s["meta"], ensure_ascii=False),
} for s in sampled]

append_to_csv(records)
